# VEYA v2 · Шаг 4 — визуализации для жюри
* **Тепловые карты (Grad-CAM)** — куда смотрит каждая нейросеть, когда ставит диагноз.
* **Карта снимков** — как ИИ «раскладывает» тысячи снимков глаза по болезням.
* **ROC-кривые** — честное качество на внутреннем и внешнем тесте.

GPU не обязателен. Картинки сохраняются в `MyDrive/veya_v2/viz/`.

## 0. Подготовка среды
Подключаем Google Drive (там хранятся данные и модели — они не пропадут, если Colab отключится) и записываем код VEYA в сессию.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
os.makedirs('/content/veya_v2', exist_ok=True)
open('/content/veya_v2/__init__.py', 'w').close()
sys.path.insert(0, '/content')
DRIVE = '/content/drive/MyDrive/veya_v2'
os.makedirs(DRIVE, exist_ok=True)

In [ ]:
!pip -q install -U timm umap-learn

In [ ]:
%%writefile /content/veya_v2/train.py
"""VEYA v2 — обучение базовых моделей ансамбля (multi-task: стадия ДР, ДР, глаукома, катаракта).

Модели («глаза» ансамбля):
  effv2s   — EfficientNetV2-S, 512px  (CNN, ловит мелкие поражения: микроаневризмы, экссудаты)
  convnext — ConvNeXt-Small, 384px    (современная CNN, другие ошибки чем у EfficientNet)
  vit      — ViT-B/14 DINOv2, 224px   (трансформер, смотрит на снимок целиком)
  retfound — RETFound ViT-L, 224px    (foundation-модель, предобучена на 1.6 млн снимков сетчатки;
                                       нужен HF_TOKEN в секретах Colab, иначе используйте vit)

Всё устойчиво к обрывам Colab: чекпоинт на Google Drive каждые N минут, запуск заново — продолжает с места.
"""
from __future__ import annotations

import json
import math
import os
import random
import shutil
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, cohen_kappa_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset, Sampler

try:
    from tqdm.auto import tqdm
except ImportError:  # pragma: no cover
    def tqdm(x, **k):
        return x

BINARY = ["dr", "glaucoma", "cataract"]
HEADS = ["dr_grade"] + BINARY

MODELS = {
    "effv2s": dict(timm="tf_efficientnetv2_s.in21k_ft_in1k", size=512, bs=16, accum=2, lr=3e-4, epochs=14,
                   samples_per_epoch=36000, channels_last=True),
    "convnext": dict(timm="convnext_small.fb_in22k_ft_in1k_384", size=384, bs=16, accum=2, lr=1.5e-4, epochs=12,
                     samples_per_epoch=36000, channels_last=True, drop_path=0.2),
    "vit": dict(timm="vit_base_patch14_dinov2.lvd142m", size=224, bs=32, accum=1, lr=4e-5, epochs=12,
                samples_per_epoch=36000, kwargs=dict(img_size=224), drop_path=0.1, layer_decay=0.75),
    "retfound": dict(timm="vit_large_patch16_224", size=224, bs=16, accum=2, lr=3e-5, epochs=10,
                     samples_per_epoch=36000, kwargs=dict(global_pool="avg"), drop_path=0.2, layer_decay=0.65,
                     hf=("YukunZhou/RETFound_mae_natureCFP", "RETFound_mae_natureCFP.pth")),
}

MEAN = np.array([0.485, 0.456, 0.406], np.float32) * 255
STD = np.array([0.229, 0.224, 0.225], np.float32) * 255


# ============================================================================ data
def load_image(path: str, size: int) -> np.ndarray:
    flag = cv2.IMREAD_REDUCED_COLOR_2 if size <= 256 else cv2.IMREAD_COLOR
    img = cv2.imread(path, flag)
    if img is None:
        img = np.zeros((size, size, 3), np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if img.shape[0] != size:
        img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    return img


def augment(img: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Аугментации, безопасные для глазного дна (не ломают признаки болезней)."""
    s = img.shape[0]
    if rng.random() < 0.5:
        img = img[:, ::-1]
    if rng.random() < 0.3:
        img = img[::-1]
    ang = rng.uniform(-180, 180)
    scale = rng.uniform(0.92, 1.12)
    tx, ty = rng.uniform(-0.04, 0.04, 2) * s
    M = cv2.getRotationMatrix2D((s / 2, s / 2), ang, scale)
    M[:, 2] += (tx, ty)
    img = cv2.warpAffine(np.ascontiguousarray(img), M, (s, s), flags=cv2.INTER_LINEAR, borderValue=(0, 0, 0))
    img = img.astype(np.float32)
    # яркость / контраст / гамма / баланс каналов (разные камеры и смартфон-насадка)
    img = img * rng.uniform(0.75, 1.25) + rng.uniform(-20, 20)
    mean = img.mean()
    img = (img - mean) * rng.uniform(0.8, 1.25) + mean
    img *= rng.uniform(0.92, 1.08, size=3)[None, None, :]
    img = np.clip(img, 0, 255)
    if rng.random() < 0.3:
        g = rng.uniform(0.8, 1.25)
        img = 255.0 * (img / 255.0) ** g
    if rng.random() < 0.2:
        k = rng.choice([3, 5])
        img = cv2.GaussianBlur(img, (k, k), 0)
    if rng.random() < 0.15:
        img = img + rng.normal(0, rng.uniform(2, 8), img.shape)
    return np.clip(img, 0, 255)


class FundusDS(Dataset):
    def __init__(self, df: pd.DataFrame, root: str, size: int, train: bool):
        self.paths = [os.path.join(root, p) for p in df["path"].values]
        self.y = df[HEADS].values.astype(np.int64)
        self.size = size
        self.train = train

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = load_image(self.paths[i], self.size)
        if self.train:
            rng = np.random.default_rng((os.getpid() * 1_000_003 + i * 7919 + int(time.time() * 1e6)) % 2**32)
            img = augment(img, rng)
        else:
            img = img.astype(np.float32)
        img = ((img - MEAN) / STD).astype(np.float32)
        x = torch.from_numpy(np.ascontiguousarray(img.transpose(2, 0, 1)))
        return x, torch.from_numpy(self.y[i])


class EpochSampler(Sampler):
    """Взвешенная выборка фиксированного размера на эпоху (баланс датасетов и редких болезней).
    Детерминирована по номеру эпохи → после обрыва Colab продолжаем с того же шага."""

    def __init__(self, weights: np.ndarray, n: int, epoch: int, start: int = 0, seed: int = 2026):
        rng = np.random.default_rng(seed + epoch)
        p = weights / weights.sum()
        self.idx = rng.choice(len(weights), size=n, replace=True, p=p)[start:]

    def __iter__(self):
        return iter(self.idx.tolist())

    def __len__(self):
        return len(self.idx)


def sample_weights(df: pd.DataFrame) -> np.ndarray:
    """Вес снимка ~ 1/sqrt(размер его «страты»): страта = датасет + комбинация меток.
    Больные глаукомой/катарактой/тяжёлой ДР попадают в батчи заметно чаще."""
    key = (df["dataset"].astype(str) + "|" + df["dr_grade"].astype(str) + "|" +
           df["glaucoma"].astype(str) + "|" + df["cataract"].astype(str))
    cnt = key.map(key.value_counts())
    w = 1.0 / np.sqrt(cnt.values.astype(np.float64))
    # дополнительно поднимаем редкие позитивы
    w *= np.where(df["glaucoma"].values == 1, 1.5, 1.0)
    w *= np.where(df["cataract"].values == 1, 2.0, 1.0)
    w *= np.where(df["dr_grade"].values >= 3, 1.5, 1.0)
    return w


# ============================================================================ model
class VeyaNet(nn.Module):
    """Backbone + общий многослойный «нейрон-мозг» модели + 4 головы."""

    def __init__(self, name: str, pretrained=True, drop=0.3, drop_path=0.0, **kw):
        super().__init__()
        import timm
        extra = dict(kw)
        if drop_path:
            extra["drop_path_rate"] = drop_path
        self.backbone = timm.create_model(name, pretrained=pretrained, num_classes=0, **extra)
        d = self.backbone.num_features
        self.neck = nn.Sequential(
            nn.Dropout(drop), nn.Linear(d, 768), nn.GELU(),
            nn.Dropout(drop / 2), nn.Linear(768, 384), nn.GELU(),
        )
        self.heads = nn.ModuleDict({
            "dr_grade": nn.Linear(384, 5),
            **{k: nn.Linear(384, 1) for k in BINARY},
        })
        self.emb_dim = d

    def forward(self, x, return_emb=False):
        f = self.backbone(x)
        h = self.neck(f)
        out = {k: (v(h) if k == "dr_grade" else v(h).squeeze(1)) for k, v in self.heads.items()}
        return (out, f) if return_emb else out


def load_retfound(model: VeyaNet, repo: str, fname: str):
    from huggingface_hub import hf_hub_download
    token = os.environ.get("HF_TOKEN")
    path = hf_hub_download(repo, fname, token=token)
    ck = torch.load(path, map_location="cpu", weights_only=False)
    sd = ck.get("model", ck.get("teacher", ck))
    sd = {k.replace("module.", "").replace("backbone.", ""): v for k, v in sd.items()}
    sd = {k: v for k, v in sd.items() if not k.startswith(("decoder", "mask_token", "head"))}
    msg = model.backbone.load_state_dict(sd, strict=False)
    print(f"RETFound загружен. missing={len(msg.missing_keys)} {msg.missing_keys[:6]} "
          f"unexpected={len(msg.unexpected_keys)} {msg.unexpected_keys[:6]}")


def build_model(key: str, pretrained=True) -> VeyaNet:
    cfg = MODELS[key]
    m = VeyaNet(cfg["timm"], pretrained=pretrained and "hf" not in cfg,
                drop_path=cfg.get("drop_path", 0.0), **cfg.get("kwargs", {}))
    if pretrained and "hf" in cfg:
        load_retfound(m, *cfg["hf"])
    return m


def param_groups(model: VeyaNet, lr: float, wd=0.05, layer_decay: float | None = None):
    """Голова учится в 10 раз быстрее; для ViT — послойное затухание lr (нижние слои меняются меньше)."""
    head = list(model.neck.parameters()) + list(model.heads.parameters())
    groups = [dict(params=head, lr=lr * 10, weight_decay=wd)]
    bb = model.backbone
    if layer_decay and hasattr(bb, "blocks"):
        n = len(bb.blocks)
        layers = {}
        for name, p in bb.named_parameters():
            if name.startswith("blocks."):
                lid = int(name.split(".")[1]) + 1
            elif name.startswith(("cls_token", "pos_embed", "patch_embed", "reg_token")):
                lid = 0
            else:
                lid = n + 1
            no_wd = p.ndim == 1 or name.endswith(".bias") or "token" in name or "pos_embed" in name
            layers.setdefault((lid, no_wd), []).append(p)
        for (lid, no_wd), ps in layers.items():
            groups.append(dict(params=ps, lr=lr * layer_decay ** (n + 1 - lid), weight_decay=0.0 if no_wd else wd))
    else:
        dec, nodec = [], []
        for name, p in bb.named_parameters():
            (nodec if p.ndim == 1 or name.endswith(".bias") else dec).append(p)
        groups += [dict(params=dec, lr=lr, weight_decay=wd), dict(params=nodec, lr=lr, weight_decay=0.0)]
    for g in groups:
        g["base_lr"] = g["lr"]
    return groups


# ============================================================================ loss + metrics
def multitask_loss(out, y, pos_weight: dict, smoothing=0.05):
    losses = {}
    yg = y[:, 0]
    m = yg >= 0
    if m.any():
        losses["dr_grade"] = F.cross_entropy(out["dr_grade"][m].float(), yg[m], label_smoothing=smoothing)
    for j, k in enumerate(BINARY, start=1):
        yy = y[:, j]
        m = yy >= 0
        if m.any():
            t = yy[m].float() * (1 - smoothing) + smoothing / 2
            losses[k] = F.binary_cross_entropy_with_logits(
                out[k][m].float(), t, pos_weight=torch.tensor(pos_weight[k], device=t.device))
    total = sum(losses.values()) if losses else out["dr"].sum() * 0
    return total, {k: float(v.detach()) for k, v in losses.items()}


def compute_metrics(logits: dict, y: np.ndarray) -> dict:
    res = {}
    yg = y[:, 0]
    m = yg >= 0
    if m.sum() > 10:
        pred = logits["dr_grade"][m].argmax(1)
        res["dr_grade_acc"] = accuracy_score(yg[m], pred)
        res["dr_grade_qwk"] = cohen_kappa_score(yg[m], pred, weights="quadratic")
    for j, k in enumerate(BINARY, start=1):
        yy = y[:, j]
        m = yy >= 0
        if m.sum() > 10 and 0 < yy[m].sum() < m.sum():
            p = 1 / (1 + np.exp(-logits[k][m]))
            res[f"{k}_auc"] = roc_auc_score(yy[m], p)
            res[f"{k}_bal_acc"] = 0.5 * ((p[yy[m] == 1] >= 0.5).mean() + (p[yy[m] == 0] < 0.5).mean())
    keys = [res.get("dr_auc"), res.get("glaucoma_auc"), res.get("cataract_auc"), res.get("dr_grade_qwk")]
    keys = [k for k in keys if k is not None]
    res["score"] = float(np.mean(keys)) if keys else 0.0
    return res


# ============================================================================ checkpoint helpers
def atomic_save(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_local = Path("/tmp") / (path.name + ".tmp")
    torch.save(obj, tmp_local)
    tmp_remote = path.with_suffix(path.suffix + ".tmp")
    shutil.copy(tmp_local, tmp_remote)
    os.replace(tmp_remote, path)
    tmp_local.unlink(missing_ok=True)


class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)
            else:
                self.shadow[k].copy_(v)

    def copy_to(self, model):
        model.load_state_dict({k: v.to(dtype=model.state_dict()[k].dtype) for k, v in self.shadow.items()})


# ============================================================================ train / eval
@torch.no_grad()
def predict(model, df, root, size, bs=32, tta=True, device="cuda", return_emb=False, workers=2):
    model.eval()
    dl = DataLoader(FundusDS(df, root, size, train=False), batch_size=bs, shuffle=False,
                    num_workers=workers, pin_memory=True)
    outs = {k: [] for k in HEADS}
    embs = []
    for x, _ in tqdm(dl, desc="predict", leave=False):
        x = x.to(device, non_blocking=True)
        views = [x, torch.flip(x, dims=[3])] if tta else [x]
        acc, eacc = None, None
        for v in views:
            with torch.autocast(device_type="cuda" if device != "cpu" else "cpu", dtype=torch.float16,
                                enabled=device != "cpu"):
                o, e = model(v, return_emb=True)
            o = {k: t.float() for k, t in o.items()}
            acc = o if acc is None else {k: acc[k] + o[k] for k in o}
            eacc = e.float() if eacc is None else eacc + e.float()
        for k in HEADS:
            outs[k].append((acc[k] / len(views)).cpu().numpy())
        if return_emb:
            embs.append((eacc / len(views)).cpu().numpy().astype(np.float16))
    res = {k: np.concatenate(v) for k, v in outs.items()}
    if return_emb:
        res["emb"] = np.concatenate(embs)
    return res


def train_model(key: str, data_dir="/content/veya_data", drive_dir="/content/drive/MyDrive/veya_v2",
                epochs: int | None = None, ckpt_minutes=12, workers=2, max_hours: float | None = None,
                quick_test=False, pretrained=True):
    cfg = dict(MODELS[key])
    if epochs:
        cfg["epochs"] = epochs
    device = "cuda" if torch.cuda.is_available() else "cpu"
    out_dir = Path(drive_dir) / "models" / key
    out_dir.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(Path(data_dir) / "labels.csv")
    tr = df[df.split == "train"].reset_index(drop=True)
    va = df[df.split == "val"].reset_index(drop=True)
    if quick_test:
        tr, va = tr.head(64), va.head(32)
        cfg.update(samples_per_epoch=32, bs=8, accum=1)
    print(f"[{key}] train={len(tr)} val={len(va)} size={cfg['size']} device={device}")

    pos_weight = {}
    for k in BINARY:
        y = tr[k][tr[k] >= 0]
        pos = max(1, int(y.sum()))
        pos_weight[k] = float(min(5.0, math.sqrt((len(y) - pos) / pos)))
    print("pos_weight:", pos_weight)

    model = build_model(key, pretrained=pretrained and not quick_test).to(device)
    if cfg.get("channels_last") and device == "cuda":
        model = model.to(memory_format=torch.channels_last)
    opt = torch.optim.AdamW(param_groups(model, cfg["lr"], layer_decay=cfg.get("layer_decay")), betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")
    ema = EMA(model, decay=0.999 if not quick_test else 0.5)
    steps_per_epoch = cfg["samples_per_epoch"] // (cfg["bs"] * cfg["accum"])
    total_steps = steps_per_epoch * cfg["epochs"]
    warmup = max(1, int(0.03 * total_steps))

    def lr_factor(step):
        if step < warmup:
            return (step + 1) / warmup
        t = (step - warmup) / max(1, total_steps - warmup)
        return 0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * t))

    state = dict(epoch=0, step_in_epoch=0, global_step=0, best=-1.0, history=[])
    last = out_dir / "last.pt"
    if last.exists():
        ck = torch.load(last, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        scaler.load_state_dict(ck["scaler"])
        ema.shadow = {k: v.float().to(device) for k, v in ck["ema"].items()}
        state = ck["state"]
        print(f"▶ Продолжаю с эпохи {state['epoch']+1}, шаг {state['step_in_epoch']} (best={state['best']:.4f})")
    if state.get("done"):
        print("Модель уже обучена. Удалите last.pt чтобы обучить заново.")
        return out_dir

    weights = sample_weights(tr)
    ds = FundusDS(tr, data_dir, cfg["size"], train=True)
    t_start = time.time()
    last_ck = time.time()

    def save_last():
        atomic_save(dict(model=model.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(),
                         ema=ema.shadow, state=state, cfg=cfg), last)

    while state["epoch"] < cfg["epochs"]:
        ep = state["epoch"]
        sampler = EpochSampler(weights, steps_per_epoch * cfg["bs"] * cfg["accum"], ep,
                               start=state["step_in_epoch"] * cfg["bs"] * cfg["accum"])
        dl = DataLoader(ds, batch_size=cfg["bs"], sampler=sampler, num_workers=workers, pin_memory=True,
                        drop_last=True, persistent_workers=False, prefetch_factor=4 if workers else None)
        model.train()
        pbar = tqdm(dl, desc=f"[{key}] epoch {ep+1}/{cfg['epochs']}", initial=state["step_in_epoch"] * cfg["accum"],
                    total=steps_per_epoch * cfg["accum"])
        run = {}
        for it, (x, y) in enumerate(pbar):
            x = x.to(device, non_blocking=True)
            if cfg.get("channels_last") and device == "cuda":
                x = x.to(memory_format=torch.channels_last)
            y = y.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda" if device == "cuda" else "cpu", dtype=torch.float16,
                                enabled=device == "cuda"):
                out = model(x)
            loss, parts = multitask_loss(out, y, pos_weight)
            scaler.scale(loss / cfg["accum"]).backward()
            for k, v in parts.items():
                run[k] = 0.98 * run.get(k, v) + 0.02 * v
            if (it + 1) % cfg["accum"] == 0:
                f = lr_factor(state["global_step"])
                for g in opt.param_groups:
                    g["lr"] = g["base_lr"] * f
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                ema.update(model)
                state["global_step"] += 1
                state["step_in_epoch"] += 1
                pbar.set_postfix({k: f"{v:.3f}" for k, v in run.items()})
                if time.time() - last_ck > ckpt_minutes * 60:
                    save_last()
                    last_ck = time.time()
                if max_hours and time.time() - t_start > max_hours * 3600:
                    save_last()
                    print(f"⏸ Лимит времени {max_hours}ч — чекпоинт сохранён, перезапустите ячейку позже.")
                    return out_dir
        # ---- конец эпохи: валидация EMA-весов
        live = {k: v.detach().clone() for k, v in model.state_dict().items()}
        ema.copy_to(model)
        logits = predict(model, va, data_dir, cfg["size"], bs=cfg["bs"] * 2, tta=False, device=device, workers=workers)
        met = compute_metrics(logits, va[HEADS].values)
        met["epoch"] = ep + 1
        met["train_loss"] = {k: round(v, 4) for k, v in run.items()}
        state["history"].append(met)
        print(f"[{key}] epoch {ep+1}: " + ", ".join(f"{k}={v:.4f}" for k, v in met.items() if isinstance(v, float)))
        if met["score"] > state["best"]:
            state["best"] = met["score"]
            atomic_save(dict(model=model.state_dict(), cfg=cfg, metrics=met, key=key), out_dir / "best.pt")
            print(f"  ★ новый лучший score={met['score']:.4f} → best.pt")
        model.load_state_dict(live)
        state["epoch"] += 1
        state["step_in_epoch"] = 0
        save_last()
        last_ck = time.time()
        (out_dir / "history.json").write_text(json.dumps(state["history"], indent=1, default=float))
    state["done"] = True
    save_last()
    print(f"[{key}] обучение завершено. best score={state['best']:.4f}")
    return out_dir


def export_predictions(key: str, data_dir="/content/veya_data", drive_dir="/content/drive/MyDrive/veya_v2",
                       splits=("val", "test", "test_ext"), workers=2):
    """Предсказания лучшей модели (с TTA) + эмбеддинги → вход для MLP-«мозга»."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    out_dir = Path(drive_dir) / "models" / key
    ck = torch.load(out_dir / "best.pt", map_location="cpu", weights_only=False)
    model = build_model(key, pretrained=False)
    model.load_state_dict(ck["model"])
    model.to(device)
    df = pd.read_csv(Path(data_dir) / "labels.csv")
    pred_dir = Path(drive_dir) / "preds"
    pred_dir.mkdir(parents=True, exist_ok=True)
    for sp in splits:
        d = df[df.split == sp].reset_index(drop=True)
        if not len(d):
            continue
        r = predict(model, d, data_dir, ck["cfg"]["size"], bs=ck["cfg"]["bs"] * 2, tta=True,
                    device=device, return_emb=True, workers=workers)
        np.savez_compressed(pred_dir / f"{key}_{sp}.npz", image_id=d["image_id"].values, **r)
        met = compute_metrics(r, d[HEADS].values)
        print(f"[{key}] {sp}: " + ", ".join(f"{k}={v:.4f}" for k, v in met.items()))
    return pred_dir

In [ ]:
%%writefile /content/veya_v2/viz.py
"""VEYA v2 — визуализации для презентации: тепловые карты (Grad-CAM), карта снимков, ROC-кривые.

Всё работает и без GPU (на CPU чуть медленнее). Картинки сохраняются в MyDrive/veya_v2/viz/.
"""
from __future__ import annotations

import os
import tarfile
from pathlib import Path

import cv2
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

matplotlib.rcParams["font.family"] = "DejaVu Sans"

BINARY = ["dr", "glaucoma", "cataract"]
NAMES_RU = {"dr": "Диабетическая ретинопатия", "glaucoma": "Глаукома", "cataract": "Катаракта"}
GRADE_RU = ["нет ДР", "ДР 1 (лёгкая)", "ДР 2 (умеренная)", "ДР 3 (тяжёлая)", "ДР 4 (пролиферативная)"]
MEAN = np.array([0.485, 0.456, 0.406], np.float32) * 255
STD = np.array([0.229, 0.224, 0.225], np.float32) * 255

BG = "#0E1116"
FG = "#E8EAED"
ACCENT = {"healthy": "#5DCAA5", "dr": "#EF9F27", "glaucoma": "#7F77DD", "cataract": "#D4537E"}


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


# ============================================================================ data helpers
def extract_some(drive_dir, local_dir, parts=("idrid", "odir", "smdg", "airogs")):
    """Распаковывает только небольшие датасеты (быстро) — для примеров тепловых карт."""
    local = Path(local_dir)
    local.mkdir(parents=True, exist_ok=True)
    for p in parts:
        flag = local / f".{p}.done"
        t = Path(drive_dir) / "data" / "parts" / f"{p}.tar"
        if flag.exists() or not t.exists():
            continue
        with tarfile.open(t) as tar:
            tar.extractall(local)
        flag.touch()
        print("распакован", t.name)
    return local


def pick_examples(labels: pd.DataFrame, preds: dict, local_dir) -> list[dict]:
    """Уверенные и ПРАВИЛЬНЫЕ примеры по каждому классу (для слайдов) из доступных снимков."""
    have = {p for p in labels["path"] if (Path(local_dir) / p).exists()}
    out = []
    df = labels.set_index("image_id")
    for split, P in preds.items():
        ids = P["image_id"]
        d = df.loc[ids].copy()
        d["avail"] = d["path"].isin(have)
        d["p_dr"] = sigmoid(P["dr"])
        d["p_gl"] = sigmoid(P["glaucoma"])
        d["p_cat"] = sigmoid(P["cataract"])
        d["grade_pred"] = P["dr_grade"].argmax(1)
        d = d[d.avail]
        # стадии ДР
        for g in range(5):
            c = d[(d.dr_grade == g) & (d.grade_pred == g)]
            if len(c):
                r = c.sort_values("p_dr", ascending=(g == 0)).iloc[0]
                out.append(dict(image_id=r.name, path=r.path, title=GRADE_RU[g], head="dr" if g else "dr",
                                prob=float(r.p_dr), split=split))
        for lab, col, head, ttl in (("glaucoma", "p_gl", "glaucoma", "Глаукома"),
                                    ("cataract", "p_cat", "cataract", "Катаракта")):
            c = d[d[lab] == 1].sort_values(col, ascending=False)
            for _, r in c.head(2).iterrows():
                out.append(dict(image_id=r.name, path=r.path, title=ttl, head=head, prob=float(r[col]), split=split))
    # убираем повторы по заголовку (берём по 1–2 на класс)
    seen, res = {}, []
    for e in out:
        k = e["title"]
        if seen.get(k, 0) < (2 if k in ("Глаукома", "Катаракта") else 1):
            res.append(e)
            seen[k] = seen.get(k, 0) + 1
    return res


# ============================================================================ Grad-CAM
def _forward_with_features(model, x):
    bb = model.backbone
    f = bb.forward_features(x)
    f.retain_grad()
    pooled = bb.forward_head(f, pre_logits=True)
    h = model.neck(pooled)
    out = {k: (v(h) if k == "dr_grade" else v(h).squeeze(1)) for k, v in model.heads.items()}
    return out, f


def _to_map(t, bb):
    if t.ndim == 3:  # ViT: (B, tokens, C) → (B, C, side, side)
        n_prefix = getattr(bb, "num_prefix_tokens", 1)
        tokens = t[:, n_prefix:]
        side = int(round(tokens.shape[1] ** 0.5))
        return tokens.transpose(1, 2).reshape(t.shape[0], -1, side, side)
    return t


def gradcam(model, img_rgb: np.ndarray, size: int, head="dr", device="cpu"):
    model.eval()
    im = cv2.resize(img_rgb, (size, size), interpolation=cv2.INTER_AREA).astype(np.float32)
    x = torch.from_numpy(((im - MEAN) / STD).transpose(2, 0, 1)).unsqueeze(0).to(device)
    with torch.enable_grad():
        out, f = _forward_with_features(model, x)
        target = out[head].sum()
        model.zero_grad(set_to_none=True)
        target.backward()
    fmap = _to_map(f.detach(), model.backbone)
    g = _to_map(f.grad, model.backbone)
    w = g.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * fmap).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=(size, size), mode="bilinear", align_corners=False)[0, 0].cpu().numpy()
    cam -= cam.min()
    cam /= cam.max() + 1e-8
    # маска круга сетчатки — чтобы не светились чёрные поля
    gray = cv2.cvtColor(im.astype(np.uint8), cv2.COLOR_RGB2GRAY)
    cam *= (gray > 12).astype(np.float32)
    return cam, float(sigmoid(out[head].detach().cpu().numpy())[0])


def overlay(img_rgb, cam, alpha=0.45):
    s = cam.shape[0]
    im = cv2.resize(img_rgb, (s, s))
    heat = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_INFERNO)[..., ::-1]
    m = np.clip((cam[..., None] - 0.1) / 0.9, 0, 1) * (alpha + 0.2)
    return (im * (1 - m) + heat * m).astype(np.uint8)


def gradcam_figure(models: dict, examples: list[dict], local_dir, out_png, device="cpu"):
    """Ряд на каждый пример: исходник + тепловая карта от каждой модели."""
    cols = 1 + len(models)
    rows = len(examples)
    fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.3 * rows), facecolor=BG)
    axes = np.atleast_2d(axes)
    for i, ex in enumerate(examples):
        img = cv2.cvtColor(cv2.imread(str(Path(local_dir) / ex["path"])), cv2.COLOR_BGR2RGB)
        axes[i, 0].imshow(cv2.resize(img, (384, 384)))
        axes[i, 0].set_title(f"{ex['title']}", color=FG, fontsize=11)
        for j, (name, (model, size)) in enumerate(models.items(), start=1):
            cam, p = gradcam(model, img, size, head=ex["head"], device=device)
            axes[i, j].imshow(overlay(img, cam))
            axes[i, j].set_title(f"{name}: {100 * p:.0f}%", color=FG, fontsize=10)
        for a in axes[i]:
            a.axis("off")
    fig.suptitle("Куда смотрит ИИ: тепловые карты внимания (Grad-CAM)", color=FG, fontsize=15, y=0.995)
    fig.tight_layout()
    fig.savefig(out_png, dpi=130, facecolor=BG)
    plt.close(fig)
    return out_png


def _consensus_cam(models: dict, img, head, device, out_size=448):
    """Средняя карта внимания всех моделей («консилиум») + средняя вероятность."""
    cams, probs = [], []
    for model, size in models.values():
        cam, p = gradcam(model, img, size, head=head, device=device)
        cams.append(cv2.resize(cam, (out_size, out_size), interpolation=cv2.INTER_LINEAR))
        probs.append(p)
    cam = cv2.GaussianBlur(np.mean(cams, 0).astype(np.float32), (0, 0), out_size / 70)
    hi = np.percentile(cam[cam > 0], 99.5) if (cam > 0).any() else 1.0
    cam = np.clip(cam / (hi + 1e-8), 0, 1) ** 0.8
    return cam, float(np.mean(probs))


def glow_overlay(img_rgb, cam):
    """Для слайда: снимок слегка затемнён, внимание ИИ «светится» поверх."""
    s = cam.shape[0]
    im = cv2.resize(img_rgb, (s, s), interpolation=cv2.INTER_AREA).astype(np.float32)
    heat = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_INFERNO)[..., ::-1].astype(np.float32)
    m = np.clip((cam[..., None] - 0.15) / 0.6, 0, 1) * 0.85
    m *= (cv2.cvtColor(im.astype(np.uint8), cv2.COLOR_RGB2GRAY) > 12)[..., None]
    base = im * 0.6
    return np.clip(base * (1 - m) + heat * m, 0, 255).astype(np.uint8)


def gradcam_slide(models: dict, examples: list[dict], local_dir, out_png, device="cpu", max_cols=6):
    """Слайд 16:9 — сверху снимок, снизу куда смотрит ИИ (консенсус всех моделей)."""
    order = ["нет ДР", "ДР 2 (умеренная)", "ДР 4 (пролиферативная)", "ДР 3 (тяжёлая)", "Глаукома", "Катаракта",
             "ДР 1 (лёгкая)"]
    chosen, used = [], set()
    for t in order:
        for e in examples:
            if e["title"] == t and t not in used:
                chosen.append(e)
                used.add(t)
                break
    chosen = sorted(chosen[:max_cols], key=lambda e: order.index(e["title"]))
    n = len(chosen)
    fig, axes = plt.subplots(2, n, figsize=(3.2 * n, 7.6), facecolor=BG, gridspec_kw=dict(hspace=0.12, wspace=0.04))
    axes = np.array(axes).reshape(2, n)
    for j, ex in enumerate(chosen):
        img = cv2.cvtColor(cv2.imread(str(Path(local_dir) / ex["path"])), cv2.COLOR_BGR2RGB)
        cam, p = _consensus_cam(models, img, ex["head"], device)
        sick = ex["title"] != "нет ДР"
        verdict = f"{100 * p:.0f}%" if sick else f"здоров · {100 * (1 - p):.0f}%"
        axes[0, j].imshow(cv2.resize(img, (448, 448), interpolation=cv2.INTER_AREA))
        axes[0, j].set_title(ex["title"], color=FG, fontsize=13, pad=8)
        axes[1, j].imshow(glow_overlay(img, cam))
        axes[1, j].set_title(f"ИИ: {verdict}", color="#EF9F27" if sick else "#5DCAA5", fontsize=13, pad=8)
        for a in axes[:, j]:
            a.axis("off")
    k = len(models)
    who = "одной нейросети" if k == 1 else f"консилиума из {k} нейросетей"
    fig.suptitle(f"Куда смотрит ИИ Veya: тепловая карта внимания {who}", color=FG, fontsize=17, y=0.99)
    fig.text(0.5, 0.015, "Светлые пятна — участки, на которых нейросеть основывает диагноз "
             "(кровоизлияния, экссудаты, диск зрительного нерва, помутнение хрусталика).",
             ha="center", color="#9AA0A6", fontsize=11)
    fig.savefig(out_png, dpi=150, facecolor=BG, bbox_inches="tight")
    plt.close(fig)
    return out_png


# ============================================================================ map of images
def embedding_map(pred_npz, labels: pd.DataFrame, out_png, max_points=15000, seed=0):
    """«Карта» снимков: внутреннее представление модели, сжатое до 2D (UMAP/t-SNE)."""
    P = np.load(pred_npz, allow_pickle=True)
    ids, emb = P["image_id"], P["emb"].astype(np.float32)
    if len(ids) > max_points:
        idx = np.random.default_rng(seed).choice(len(ids), max_points, replace=False)
        ids, emb = ids[idx], emb[idx]
    d = labels.set_index("image_id").loc[ids]
    from sklearn.decomposition import PCA
    z = PCA(min(50, len(emb) - 1, emb.shape[1]), random_state=seed).fit_transform(emb)
    try:
        import umap
        xy = umap.UMAP(n_neighbors=30, min_dist=0.15, random_state=seed).fit_transform(z)
        method = "UMAP"
    except Exception:
        from sklearn.manifold import TSNE
        xy = TSNE(2, perplexity=min(40, max(5, len(z) // 4)), init="pca", random_state=seed).fit_transform(z)
        method = "t-SNE"
    cat = np.full(len(d), "other", dtype=object)
    cat[(d.dr == 0).values & (d.glaucoma != 1).values & (d.cataract != 1).values] = "healthy"
    cat[(d.dr == 1).values] = "dr"
    cat[(d.glaucoma == 1).values] = "glaucoma"
    cat[(d.cataract == 1).values] = "cataract"
    fig, ax = plt.subplots(figsize=(11, 9), facecolor=BG)
    ax.set_facecolor(BG)
    ax.scatter(xy[cat == "other", 0], xy[cat == "other", 1], s=2, c="#444441", alpha=0.3, linewidths=0)
    labels_ru = {"healthy": "Здоровые", "dr": "Диабетическая ретинопатия", "glaucoma": "Глаукома", "cataract": "Катаракта"}
    for k in ["healthy", "dr", "glaucoma", "cataract"]:
        m = cat == k
        ax.scatter(xy[m, 0], xy[m, 1], s=4 if k in ("healthy", "dr") else 9, c=ACCENT[k], alpha=0.7,
                   linewidths=0, label=f"{labels_ru[k]} ({m.sum()})")
    ax.legend(loc="lower left", frameon=False, labelcolor=FG, fontsize=12, markerscale=4)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title(f"Как ИИ «видит» {len(d):,} снимков глаза — карта ({method}).\n"
                 "Каждая точка — снимок; похожие для нейросети снимки оказываются рядом.",
                 color=FG, fontsize=14)
    fig.tight_layout()
    fig.savefig(out_png, dpi=140, facecolor=BG)
    plt.close(fig)
    return out_png


# ============================================================================ ROC
def roc_figure(pred_npzs: dict, labels: pd.DataFrame, out_png, title):
    from sklearn.metrics import roc_auc_score, roc_curve
    df = labels.set_index("image_id")
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), facecolor=BG)
    for ax, k in zip(axes, BINARY):
        ax.set_facecolor(BG)
        for name, f in pred_npzs.items():
            if not Path(f).exists():
                continue
            P = np.load(f, allow_pickle=True)
            y = df.loc[P["image_id"], k].values
            m = y >= 0
            if m.sum() < 20 or y[m].min() == y[m].max():
                continue
            p = sigmoid(P[k][m])
            fpr, tpr, _ = roc_curve(y[m], p)
            ax.plot(fpr, tpr, lw=2, label=f"{name}: AUC {roc_auc_score(y[m], p):.3f}")
        ax.plot([0, 1], [0, 1], ls="--", c="#5F5E5A", lw=1)
        ax.set_title(NAMES_RU[k], color=FG, fontsize=13)
        ax.set_xlabel("Ложные тревоги (1 − специфичность)", color=FG)
        ax.set_ylabel("Найдено больных (чувствительность)", color=FG)
        ax.tick_params(colors=FG)
        for s in ax.spines.values():
            s.set_color("#5F5E5A")
        ax.legend(frameon=False, labelcolor=FG, loc="lower right")
    fig.suptitle(title, color=FG, fontsize=15)
    fig.tight_layout()
    fig.savefig(out_png, dpi=130, facecolor=BG)
    plt.close(fig)
    return out_png

## 1. Тепловые карты

In [ ]:
import shutil, glob, torch, pandas as pd, numpy as np
from pathlib import Path
from IPython.display import Image, display
os.makedirs(f'{DRIVE}/code', exist_ok=True)
for f in glob.glob('/content/veya_v2/*.py'): shutil.copy(f, f'{DRIVE}/code/')
from veya_v2 import train as T, viz as V
import importlib; importlib.reload(V)
LOCAL = '/content/veya_data'
V.extract_some(DRIVE, LOCAL)
labels = pd.read_csv(f'{DRIVE}/data/labels.csv')
OUT = f'{DRIVE}/viz'; os.makedirs(OUT, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
names = {'effv2s': 'EfficientNetV2', 'convnext': 'ConvNeXt', 'vit': 'ViT', 'retfound': 'RETFound'}
models = {}
for k in names:
    f = Path(f'{DRIVE}/models/{k}/best.pt')
    if not f.exists() or not Path(f'{DRIVE}/preds/{k}_test.npz').exists(): continue
    ck = torch.load(f, map_location='cpu', weights_only=False)
    m = T.build_model(k, pretrained=False); m.load_state_dict(ck['model']); m.to(device).eval()
    models[names[k]] = (m, ck['cfg']['size'])
print('модели:', list(models))
main = [k for k in names if Path(f'{DRIVE}/preds/{k}_test.npz').exists()][0]
preds = {sp: dict(np.load(f'{DRIVE}/preds/{main}_{sp}.npz', allow_pickle=True)) for sp in ['test_ext', 'test']}
ex = V.pick_examples(labels, preds, LOCAL)
print([e['title'] for e in ex])
display(Image(V.gradcam_slide(models, ex, LOCAL, f'{OUT}/gradcam_slide.png', device=device)))
display(Image(V.gradcam_figure(models, ex, LOCAL, f'{OUT}/gradcam.png', device=device)))

## 2. Карта снимков

In [ ]:
display(Image(V.embedding_map(f'{DRIVE}/preds/{main}_val.npz', labels, f'{OUT}/map.png')))

## 3. ROC-кривые

In [ ]:
for sp, ttl in [('test', 'Внутренний тест (новые пациенты)'), ('test_ext', 'Внешний тест (чужие клиники)')]:
    files = {names[k]: f'{DRIVE}/preds/{k}_{sp}.npz' for k in names}
    display(Image(V.roc_figure(files, labels, f'{OUT}/roc_{sp}.png', ttl)))

Все картинки лежат в `MyDrive/veya_v2/viz/`.